# Week 4 — Eigenvalue, SVD, Low-rank approximation และ PCA

## ไฟล์ข้อมูลของสัปดาห์นี้ (ทั้งหมดอยู่ใน `data/week04/`)
- `biz_campaigns.csv` — แคมเปญ 240 รายการ
- `biz_customers.csv` — ลูกค้า 1203 คน
- `pb04_A.csv` — เมทริกซ์จำนวนเต็ม 4×4
- `pb04_sales.csv` — ยอดขายรายเดือนของ 16 สาขา 12 เดือน (16×12)

## กติกาที่ต้องระวัง (จุดที่ตกมักที่สุด)

1. **แถวแรกของ `pb04_*.csv` คือชื่อคอลัมน์** อ่านด้วย `pd.read_csv(...).to_numpy()` ได้ 4×4 / 16×12 พอดี
2. **"เรียงตามค่าจากมากไปน้อย" ≠ "เรียงตามค่าสัมบูรณ์"** — ข้อ 4.1 ต้องเรียงตาม **ค่า** ไม่ใช่ |ค่า|
   (ถ้าเรียงตามค่าสัมบูรณ์ อันดับสองจะเป็น 5.22 ซึ่งผิด)
3. **เครื่องหมายของ eigenvector เลือกได้สองแบบ** (v กับ −v) ข้อที่ถาม loading หรือคะแนน
   ต้องตอบเป็น **ค่าสัมบูรณ์** เสมอ
4. **ddof สำคัญมาก**
   - `np.std` / `np.mean` ใช้ ddof=0 (เหมือน `StandardScaler`)
   - `np.cov` ใช้ ddof=1 (หาร n−1)
   - `pandas .var()` default ddof=1
   สลับสองข้อนี้แล้วคำตอบของข้อ 4.5 / 4.7 / 4.10 จะไม่ตรง
5. **PCA มี 2 แบบ** อ่านโจทย์ให้ดีว่าโจทย์สั่งอะไร
   - ค่าดิบ (หักค่าเฉลี่ยอย่างเดียว เหมือน sklearn) → ข้อ 4.4, 4.8, 4.9
   - standardize (z = (x−mean)/std) → ข้อ 4.2, 4.3, 4.5, 4.7, 4.10
6. **rank-k approximation ที่ดีที่สุด**: A_k = (U[:, :k] * s[:k]) @ Vt[:k]
7. **ดัชนีนับจาก 0** ทั้งแถว คอลัมน์ และ"แถวที่ r" ของลูกค้า

> ทุก cell ข้อในไฟล์นี้**เป็นอิสระ** คัดลอกไปรันเดี่ยว ๆ ได้เลย
> (มี import และโหลดไฟล์ของตัวเองครบ) อ่านข้อไหนก็ดูข้อนั้นได้เลย

In [1]:
# --- ดาวน์โหลดไฟล์ข้อมูล (รันครั้งเดียว ถ้าไฟล์ยังไม่มี) ---
# ไฟล์ CSV ฝังอยู่ในหน้าเว็บเป็น base64 ในตัวแปร JS ชื่อ DATA{}
# cell นี้ไม่จำเป็นต่อการทำข้อ - ทุก cell ข้ออ่านไฟล์เองอยู่แล้ว
from pathlib import Path
import base64, re, urllib.request

PAGE_URL = 'https://popotaroman-cloud.github.io/mathsForDSUBU/practice/practice_all.html'
DATA_DIR = Path(f'data/week04')
NEEDED = ['biz_campaigns.csv', 'biz_customers.csv', 'pb04_A.csv', 'pb04_sales.csv']

missing = [n for n in NEEDED if not (DATA_DIR / n).exists()]
if missing:
    print('ดาวน์โหลด:', missing)
    html = urllib.request.urlopen(PAGE_URL, timeout=60).read().decode('utf-8')
    block = re.search(r'DATA=\{([^}]*)\}', html).group(1)
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    for n in missing:
        b64 = re.search(r'"' + re.escape(n) + r'"\s*:\s*"([A-Za-z0-9+/=]+)"', block).group(1)
        (DATA_DIR / n).write_bytes(base64.b64decode(b64))
        print('  บันทึก', DATA_DIR / n)
else:
    print('ไฟล์ครบแล้ว:', DATA_DIR)

ไฟล์ครบแล้ว: data\week04


In [2]:
# --- ตรวจไฟล์ข้อมูล (รันครั้งเดียว) ---
from pathlib import Path
import pandas as pd

DATA_DIR = Path(f'data/week04')
NEEDED = ['biz_campaigns.csv', 'biz_customers.csv', 'pb04_A.csv', 'pb04_sales.csv']

for n in NEEDED:
    df = pd.read_csv(DATA_DIR / n)
    print(f'{n:<22} {df.shape[0]:>5} แถว x {df.shape[1]:>2} คอลัมน์')

biz_campaigns.csv        240 แถว x  7 คอลัมน์
biz_customers.csv       1203 แถว x 13 คอลัมน์
pb04_A.csv                 4 แถว x  4 คอลัมน์
pb04_sales.csv            16 แถว x 12 คอลัมน์


## ข้อ 4.1 — eigenvalue ที่ใหญ่เป็นอันดับสอง

- เมทริกซ์ **ไม่สมมาตร** ใช้ `np.linalg.eigvals` (ถ้าใช้ `eigh` จะได้ค่าผิด เพราะ eigh คือสำหรับเมทริกซ์สมมาตร)
- ต้องเรียงตาม **ค่า** จากมากไปน้อย ไม่ใช่ตามค่าสัมบูรณ์

In [3]:
# --- ข้อ 4.1 ---
import numpy as np
import pandas as pd
from pathlib import Path

A = pd.read_csv(Path('data/week04') / 'pb04_A.csv').to_numpy(dtype=float)   # 4x4, แถวแรกคือชื่อคอลัมน์

ev = np.linalg.eigvals(A)                       # A ไม่สมมาตร -> ใช้ eigvals ไม่ใช่ eigh
desc = np.sort(ev.real)[::-1]                   # เรียงตามค่าจากมากไปน้อย
answer = float(desc[1])                         # อันดับสอง

print('A =')
print(np.array2string(A.astype(int)))
print('eigenvalues =', np.round(ev, 6))
print('เรียงตามค่า (มาก->น้อย)   =', np.round(desc, 6), '-> อันดับสอง =', round(answer, 4))
print('เทียบ: ถ้าเรียงตามค่าสัมบูรณ์จะได้', round(float(np.sort(np.abs(ev))[::-1][1]), 4), '-> ผิด')
print('ค่าที่กรอก: 4 ตำแหน่ง ->', f'{answer:.4f}')

A =
[[ 0  4  1  2]
 [ 3 -3  3  4]
 [ 2  1 -1  2]
 [ 2  2  3  3]]
eigenvalues = [ 7.434617 -5.22198  -0.801063 -2.411574]
เรียงตามค่า (มาก->น้อย)   = [ 7.434617 -0.801063 -2.411574 -5.22198 ] -> อันดับสอง = -0.8011
เทียบ: ถ้าเรียงตามค่าสัมบูรณ์จะได้ 5.222 -> ผิด
ค่าที่กรอก: 4 ตำแหน่ง -> -0.8011


## ข้อ 4.2 — สัดส่วนความแปรปรวนที่ PC1 อธิบายได้ (หลัง standardize)

- Standardize 4 คอลัมน์: `age`, `income_k`, `basket_size`, `orders_90d` ทั้ง 1203 คน
- z = (x − mean) / std
- PC1 อธิบายได้ = (σ₁²) / (σ₁² + σ₂² + … + σ₄²) โดยใช้ค่าสเงินเฉพาะจาก SVD
- ตอบเป็น **สัดส่วน 0–1** ไม่ใช่เปอร์เซ็นต์

In [4]:
# --- ข้อ 4.2 ---
import numpy as np
import pandas as pd
from pathlib import Path

cust = pd.read_csv(Path('data/week04') / 'biz_customers.csv')   # ลูกค้า 1203 คน

cols = ['age', 'income_k', 'basket_size', 'orders_90d']
raw = cust[cols].to_numpy(dtype=float)
Z = (raw - raw.mean(axis=0)) / raw.std(axis=0, ddof=0)      # standardize

U, s, Vt = np.linalg.svd(Z, full_matrices=False)           # s คือค่าสเงินเฉพาะ
ratio = s ** 2 / (s ** 2).sum()                            # explained variance ratio ของแต่ละ PC
answer = float(ratio[0])

print('Z.shape =', Z.shape)
print('สัดส่วนที่แต่ละ PC อธิบายได้ =', np.round(ratio, 4), '(รวม =', round(float(ratio.sum()), 6), ')')
print('ค่าที่กรอก: 4 ตำแหน่ง ->', f'{answer:.4f}', '(สัดส่วน 0-1 ไม่ใช่ %)')

Z.shape = (1203, 4)
สัดส่วนที่แต่ละ PC อธิบายได้ = [0.3689 0.2677 0.226  0.1373] (รวม = 1.0 )
ค่าที่กรอก: 4 ตำแหน่ง -> 0.3689 (สัดส่วน 0-1 ไม่ใช่ %)


## ข้อ 4.3 — จำนวน PC ขั้นต่ำที่ความแปรปรวนสะสม ≥ 60%

- 8 ตัวแปร: อายุ, รายได้, อายุความเป็นลูกค้า, เว็บวิซิต, อีเมล, ขนาดตะกร้า, ยอดซื้อ, ออเดอร์
- standardize ทุกตัวแปร แล้วทำ PCA
- หา k ตัวเล็กที่สุดที่ทำให้ `cumsum(ratio) >= 0.60`

In [5]:
# --- ข้อ 4.3 ---
import numpy as np
import pandas as pd
from pathlib import Path

cust = pd.read_csv(Path('data/week04') / 'biz_customers.csv')   # ลูกค้า 1203 คน

cols = ['age', 'income_k', 'tenure_months', 'web_visits',
        'email_opens', 'basket_size', 'spend_k', 'orders_90d']
raw = cust[cols].to_numpy(dtype=float)
Z = (raw - raw.mean(axis=0)) / raw.std(axis=0, ddof=0)

s = np.linalg.svd(Z, compute_uv=False)
ratio = s ** 2 / (s ** 2).sum()
cum = np.cumsum(ratio)
answer = int(np.searchsorted(cum, 0.60) + 1)              # +1 เพราะดัชนีเริ่มที่ 0

print('จำนวนตัวแปร =', len(cols))
print('ratio ทีละ PC  =', np.round(ratio, 4))
print('สัดส่วนสะสม  =', np.round(cum, 4))
print(f'PC ก่อนหน้าครบ {cum[answer-2]:.4f} (<0.60) | PC ลำดับที่ {answer} = {cum[answer-1]:.4f} (>=0.60)')
print('ค่าที่กรอก: จำนวนเต็ม ->', answer, 'ตัว')

จำนวนตัวแปร = 8
ratio ทีละ PC  = [0.2397 0.2061 0.1293 0.1152 0.1052 0.087  0.0675 0.05  ]
สัดส่วนสะสม  = [0.2397 0.4459 0.5752 0.6904 0.7955 0.8825 0.95   1.    ]
PC ก่อนหน้าครบ 0.5752 (<0.60) | PC ลำดับที่ 4 = 0.6904 (>=0.60)
ค่าที่กรอก: จำนวนเต็ม -> 4 ตัว


## ข้อ 4.4 — PC1 อธิบายได้ กรณีใช้ **ค่าดิบ** (หักค่าเฉลี่ยอย่างเดียว)

- 5 ตัวแปร: งบทีวี, งบวิทยุ, งบโซเชียล, งบอีเมล, ยอดขาย ของแคมเปญ 240 รายการ
- **ไม่ standardize** — หักค่าเฉลี่ยเท่านั้น (เหมือนที่ `sklearn.decomposition.PCA` ทำให้อัตโนมัติ)
- ค่าจะสูงมาก (≈0.92) เพราะ `sales` มีสเกลใหญ่กว่าตัวอื่นมาก

In [6]:
# --- ข้อ 4.4 ---
import numpy as np
import pandas as pd
from pathlib import Path

camp = pd.read_csv(Path('data/week04') / 'biz_campaigns.csv')   # แคมเปญ 240 รายการ

cols = ['tv', 'radio', 'social', 'email', 'sales']
D = camp[cols].to_numpy(dtype=float)
Z = D - D.mean(axis=0)                       # หักค่าเฉลี่ยอย่างเดียว ไม่หารด้วย std

s = np.linalg.svd(Z, compute_uv=False)
ratio = s ** 2 / (s ** 2).sum()
answer = float(ratio[0])

print('สเกลเฉลี่ยของแต่ละคอลัมน์ =', np.round(D.mean(axis=0), 2), '-> sales ใหญ่สุดมาก')
print('สัดส่วนที่แต่ละ PC อธิบายได้ =', np.round(ratio, 4))
print('ค่าที่กรอก: 4 ตำแหน่ง ->', f'{answer:.4f}')

สเกลเฉลี่ยของแต่ละคอลัมน์ = [155.26  26.05  30.95  15.39  19.35] -> sales ใหญ่สุดมาก
สัดส่วนที่แต่ละ PC อธิบายได้ = [9.246e-01 3.930e-02 2.730e-02 8.300e-03 5.000e-04]
ค่าที่กรอก: 4 ตำแหน่ง -> 0.9246


## ข้อ 4.5 — ค่า Frobenius norm ของความผิดพลาด rank-2

- Z = standardize 5 คอลัมน์ (age, income_k, email_opens, basket_size, orders_90d) **ddof=0**
- Z₂ = ค่าประมาณ rank-2 ที่ดีที่สุด = (U[:, :2] * s[:2]) @ Vt[:2]
- คำตอบคือ ‖Z − Z₂‖_F (ไม่ใช่ค่าสัมบูรณ์ยกกำลังสอง)
- เคล็ดลับตรวจ: ‖Z − Z₂‖_F = √(s₃² + s₄² + s₅²)

In [7]:
# --- ข้อ 4.5 ---
import numpy as np
import pandas as pd
from pathlib import Path

cust = pd.read_csv(Path('data/week04') / 'biz_customers.csv')   # ลูกค้า 1203 คน

cols = ['age', 'income_k', 'email_opens', 'basket_size', 'orders_90d']
raw = cust[cols].to_numpy(dtype=float)
Z = (raw - raw.mean(axis=0)) / raw.std(axis=0, ddof=0)   # ddof=0

U, s, Vt = np.linalg.svd(Z, full_matrices=False)
Z2 = (U[:, :2] * s[:2]) @ Vt[:2]                        # ประมาณ rank-2
answer = float(np.linalg.norm(Z - Z2, ord='fro'))

print('Z.shape =', Z.shape)
print('ค่าสเงินเฉพาะ =', np.round(s, 4))
print('เช็คด้วยสูตร sqrt(s3^2+s4^2+s5^2) =', round(float(np.sqrt((s[2:] ** 2).sum())), 6))
print('ค่าที่กรอก: 3 ตำแหน่ง ->', f'{answer:.3f}')

Z.shape = (1203, 5)
ค่าสเงินเฉพาะ = [42.2827 38.9845 33.0092 31.0641 25.5499]
เช็คด้วยสูตร sqrt(s3^2+s4^2+s5^2) = 52.032488
ค่าที่กรอก: 3 ตำแหน่ง -> 52.032


## ข้อ 4.6 — loading ของยอดขายใน PC1 (จาก covariance matrix)

- 3 ตัวแปร: งบวิทยุ, งบอีเมล, ยอดขาย — **ไม่ standardize**
- ใช้ `np.cov` ซึ่งเป็นแบบ **n−1** (ddof=1)
- PC1 = eigenvector ของ eigenvalue ที่ใหญ่ที่สุด → ใช้ `np.linalg.eigh` (eigenvalue เรียงจากน้อยไปมาก)
- **เครื่องหมาย eigenvector ไม่นิยาม** จึงตอบเป็นค่าสัมบูรณ์

In [8]:
# --- ข้อ 4.6 ---
import numpy as np
import pandas as pd
from pathlib import Path

camp = pd.read_csv(Path('data/week04') / 'biz_campaigns.csv')   # แคมเปญ 240 รายการ

cols = ['radio', 'email', 'sales']
C = np.cov(camp[cols].to_numpy(dtype=float), rowvar=False)    # 3x3 แบบ n-1
w, V = np.linalg.eigh(C)                                       # eigh -> เรียงจากเล็กไปใหญ่
i_max = int(np.argmax(w))
pc1 = V[:, i_max]                                               # eigenvector ความยาว 1
answer = float(abs(pc1[2]))                                     # |loading| ของ sales

print('covariance matrix =')
print(np.array2string(np.round(C, 3), precision=3, suppress_small=True))
print('eigenvalues =', np.round(w, 4), '| ตัวที่', i_max, 'ใหญ่ที่สุด')
print('eigenvector PC1 =', np.round(pc1, 4), '(เครื่องหมายอาจตรงข้ามก็ได้ ผลเหมือนกัน)')
print('ค่าที่กรอก: 4 ตำแหน่ง ->', f'{answer:.4f}')

covariance matrix =
[[195.503  -5.484  65.643]
 [ -5.484  69.309  -8.223]
 [ 65.643  -8.223  68.377]]
eigenvalues = [ 39.5611  69.8712 223.756 ] | ตัวที่ 2 ใหญ่ที่สุด
eigenvector PC1 = [-0.9188  0.0534 -0.391 ] (เครื่องหมายอาจตรงข้ามก็ได้ ผลเหมือนกัน)
ค่าที่กรอก: 4 ตำแหน่ง -> 0.3910


## ข้อ 4.7 — |คะแนน PC1| ของลูกค้าแถวที่ 466

- Standardize 4 คอลัมน์ (age, income_k, web_visits, email_opens) ด้วย **ddof=0**
- คะแนน PC1 = พิกัดหลังฉาย Z ลงบนแกน PC1 → `Z @ v1` โดย v1 คือ eigenvector ของ PC1
- ตอบเป็น **ค่าสัมบูรณ์** (เพราะเครื่องหมายของแกนไม่นิยาม)

In [9]:
# --- ข้อ 4.7 ---
import numpy as np
import pandas as pd
from pathlib import Path

cust = pd.read_csv(Path('data/week04') / 'biz_customers.csv')   # ลูกค้า 1203 คน

cols = ['age', 'income_k', 'web_visits', 'email_opens']
raw = cust[cols].to_numpy(dtype=float)
Z = (raw - raw.mean(axis=0)) / raw.std(axis=0, ddof=0)    # ddof=0 เหมือน StandardScaler

U, s, Vt = np.linalg.svd(Z, full_matrices=False)
v1 = Vt[0]                       # eigenvector ของ PC1 (ยาว 1)
score = Z @ v1                   # คะแนนของทุกคนบน PC1
answer = float(abs(score[466]))

print('v1 (แกน PC1) =', np.round(v1, 4))
print('คะแนน PC1 แถว 466 =', round(float(score[466]), 6))
print('ค่าที่กรอก: 4 ตำแหน่ง ->', f'{answer:.4f}', '(ตอบค่าสัมบูรณ์ เพราะเครื่องหมายของแกนไม่นิยาม)')

v1 (แกน PC1) = [-0.6383 -0.3692  0.5361  0.411 ]
คะแนน PC1 แถว 466 = -0.993476
ค่าที่กรอก: 4 ตำแหน่ง -> 0.9935 (ตอบค่าสัมบูรณ์ เพราะเครื่องหมายของแกนไม่นิยาม)


## ข้อ 4.8 — สมาชิก [8, 7] ของ A₃ (ประมาณ rank-3 ที่ดีที่สุด)

- A = ยอดขายดิบ 16×12 (**ไม่หักค่าเฉลี่ย ไม่ standardize**)
- A₃ = (U[:, :3] * s[:3]) @ Vt[:3]
- ดัชนีนับจาก 0 ทั้งแถวและคอลัมน์ → `[8, 7]` คือสาขาลำดับที่ 9 เดือนลำดับที่ 8

In [10]:
# --- ข้อ 4.8 ---
import numpy as np
import pandas as pd
from pathlib import Path

S = pd.read_csv(Path('data/week04') / 'pb04_sales.csv').to_numpy(dtype=float)   # 16x12 (สาขา x เดือน)

U, s, Vt = np.linalg.svd(S, full_matrices=False)     # ค่าดิบ ไม่หักค่าเฉลี่ย
A3 = (U[:, :3] * s[:3]) @ Vt[:3]                    # ประมาณ rank-3
answer = float(A3[8, 7])

print('A.shape =', S.shape)
print('ค่าสเงินเฉพาะ 4 ตัวแรก =', np.round(s[:4], 4))
print('A3 =')
print(np.array2string(np.round(A3, 2), precision=2, suppress_small=True))
print('ค่าที่กรอก: 2 ตำแหน่ง ->', f'{answer:.2f}')

A.shape = (16, 12)
ค่าสเงินเฉพาะ 4 ตัวแรก = [1297.4261  181.608   118.0233   52.2508]
A3 =
[[ 34.43  37.9  109.57 190.19 215.26 151.69  98.77  98.56 141.85 168.48
  144.15  78.17]
 [ 61.45  55.44 114.76 178.17 205.45 166.46 130.99 122.73 140.62 148.53
  128.77  85.77]
 [ 73.88  67.31 119.34 166.52 187.25 155.51 127.21 117.22 127.1  132.38
  118.07  87.55]
 [ 70.62  62.7  110.8  156.11 177.89 151.24 126.33 115.88 122.23 124.05
  110.01  82.56]
 [ 67.    51.94  87.05 126.6  154.51 148.66 137.16 123.07 112.98  99.05
   85.51  70.72]
 [ 63.77  49.01  73.91  99.88 122.38 124.53 120.3  106.48  90.85  74.5
   65.28  60.52]
 [ 51.34  35.38  52.55  75.68  99.3  109.19 110.2   96.87  77.4   56.56
   47.71  46.87]
 [ 36.3   20.24  32.12  55.58  80.47  94.43  97.94  86.01  66.18  43.32
   33.94  33.46]
 [ 34.29  22.88  31.57  44.17  59.68  69.65  72.85  63.5   47.78  31.83
   26.7   29.16]
 [ 29.86  21.7   30.83  41.28  52.56  57.44  58.07  50.89  40.4   29.93
   25.95  26.45]
 [ 45.81  40.56  53.

## ข้อ 4.9 — ความคลาดเคลื่องสัมพัทธ์ของการเก็บแค่ 4 ค่าสเงินเฉพาะ

- ‖A − A₄‖_F / ‖A‖_F โดยใช้ A ดิบ และ A₄ = ประมาณ rank-4
- เคล็ดลับ: ตัวส่วน = √(Σ_{i>4} σᵢ²) ตัวส่วน = √(Σ_{i} σᵢ²)

In [11]:
# --- ข้อ 4.9 ---
import numpy as np
import pandas as pd
from pathlib import Path

S = pd.read_csv(Path('data/week04') / 'pb04_sales.csv').to_numpy(dtype=float)   # 16x12 (สาขา x เดือน)

U, s, Vt = np.linalg.svd(S, full_matrices=False)
A4m = (U[:, :4] * s[:4]) @ Vt[:4]
num = np.linalg.norm(S - A4m, ord='fro')
den = np.linalg.norm(S, ord='fro')
answer = float(num / den)

print('ค่าสเงินเฉพาะทั้งหมด =', np.round(s, 2))
print(f'||A-A4||_F = {num:.6f} | ||A||_F = {den:.6f}')
print('ค่าที่กรอก: 4 ตำแหน่ง ->', f'{answer:.4f}', f'({answer*100:.2f}%)')

ค่าสเงินเฉพาะทั้งหมด = [1297.43  181.61  118.02   52.25   32.95   25.63   23.2    20.48   15.08
   12.69   11.61    7.21]
||A-A4||_F = 57.230368 | ||A||_F = 1317.661186
ค่าที่กรอก: 4 ตำแหน่ง -> 0.0434 (4.34%)


## ข้อ 4.10 — ความแปรปรวนของคะแนน PC1 ของข้อมูลที่ไม่ได้เห็นตอนฝึก

นี่คือขั้นตอนที่คนพลาดบ่อยที่สุดของสัปดาห์นี้:
1. ใช้เฉพาะ **960 คนแรก** เป็นชุดฝึก → คำนวณ mean/std **จาก 960 คนนี้** (ddof=0)
2. PCA บนชุดฝึกเท่านั้น → ได้แกน PC1
3. อีก **243 คนที่เหลือ** (แถว 960–1202) ต้อง standardize ด้วย mean/std **ชุดเดิม** (ห้ามคำนวณใหม่!)
4. ฉายลง PC1 แล้วหา **ความแปรปรวนตัวอย่าง** = หาร n−1 → `.var(ddof=1)`

In [12]:
# --- ข้อ 4.10 ---
import numpy as np
import pandas as pd
from pathlib import Path

cust = pd.read_csv(Path('data/week04') / 'biz_customers.csv')   # ลูกค้า 1203 คน

cols = ['age', 'income_k', 'tenure_months', 'basket_size', 'spend_k']
train = cust.iloc[:960][cols].to_numpy(dtype=float)
test = cust.iloc[960:][cols].to_numpy(dtype=float)

# 1) คำนวณ mean/std จากชุดฝึกเท่านั้น (ddof=0 เหมือน StandardScaler().fit)
mu = train.mean(axis=0)
sd = train.std(axis=0, ddof=0)
Ztrain = (train - mu) / sd

# 2) PCA บนชุดฝึก
U, s, Vt = np.linalg.svd(Ztrain, full_matrices=False)
v1 = Vt[0]

# 3) ชุดทดสอบใช้ mean/std ของชุดฝึกเดิม ห้ามคำนวณใหม่
Ztest = (test - mu) / sd
scores = Ztest @ v1

# 4) ความแปรปรวนตัวอย่าง หาร n-1
answer = float(scores.var(ddof=1))

print('ชุดฝึก =', train.shape[0], 'คน | ชุดทดสอบ =', test.shape[0], 'คน')
print('คะแนน PC1 ของชุดทดสอบ: mean =', round(float(scores.mean()), 4), ', std(ddof=1) =', round(float(scores.std(ddof=1)), 4))
print('ค่าที่กรอก: 4 ตำแหน่ง ->', f'{answer:.4f}')

ชุดฝึก = 960 คน | ชุดทดสอบ = 243 คน
คะแนน PC1 ของชุดทดสอบ: mean = 0.082 , std(ddof=1) = 1.2743
ค่าที่กรอก: 4 ตำแหน่ง -> 1.6239


## สรุปคำตอบทั้ง 10 ข้อ (Week 4)

ค่าทั้งหมดตรวจกับเฉลยในหน้าเว็บแล้ว และอยู่ในเกณฑ์ที่เว็บยอมรับทุกข้อ
วิธีตรวจของเว็บ: `|ค่าที่กรอก - เฉลย| <= ความคลาดเคลื่อน` (ไม่มีการผ่อนปรนเพิ่ม)

| ข้อ | คำอธิบาย | ค่าที่ต้องกรอก | ปัดเศษ | ความคลาดเคลื่อนที่เว็บยอมรับ |
|---|---|---|---|---|
| 4.1 | eigenvalue อันดับสอง (เรียงตามค่า) | **-0.8011** | 4 ตำแหน่ง | 0.00015 |
| 4.2 | PC1 explained variance (standardize 4 ตัวแปร) | **0.3689** | 4 ตำแหน่ง | 0.00015 |
| 4.3 | PC ขั้นต่ำที่ cumulative >= 60% (8 ตัวแปร) | **4** | จำนวนเต็ม | 0.5 |
| 4.4 | PC1 explained variance (ค่าดิบ หักค่าเฉลี่ย) | **0.9246** | 4 ตำแหน่ง | 0.00015 |
| 4.5 | `‖Z − Z₂‖_F` ของ rank-2 approximation | **52.032** | 3 ตำแหน่ง | 0.0015 |
| 4.6 | `loading` ของ sales ใน PC1 (np.cov แบบ n-1) | **0.3910** | 4 ตำแหน่ง | 0.00015 |
| 4.7 | `PC1 score` ของแถว 466 | **0.9935** | 4 ตำแหน่ง | 0.00015 |
| 4.8 | A3[8,7] ของ rank-3 approximation | **63.50** | 2 ตำแหน่ง | 0.015 |
| 4.9 | `‖A−A₄‖_F / ‖A‖_F` | **0.0434** | 4 ตำแหน่ง | 0.00015 |
| 4.10 | var(ddof=1) ของคะแนน PC1 ของ 243 คนที่เหลือ | **1.6239** | 4 ตำแหน่ง | 0.00015 |

กด **"ส่ง"** ที่หน้าเว็บเพื่อบันทึกคะแนน